# 06 LangGraph状态路由与Checkpoint

**用途：** 建立临时LangGraph，查看State、节点路由、checkpoint和跨实例恢复。

> 使用方式：按顺序运行。出现 `PASS` 才代表本节验收成功；断言失败时先阅读紧邻的“失败定位”。默认不调用真实模型、不写生产数据库。

In [1]:
from pathlib import Path
import importlib.util
import json
import os
import sys
import tempfile

cwd = Path.cwd().resolve()
DAY1_ROOT = None
PROJECT2_ROOT = None
for candidate in [cwd, *cwd.parents]:
    if (candidate / "project2" / "agent_graph.py").exists():
        DAY1_ROOT = candidate
        PROJECT2_ROOT = candidate / "project2"
        break
    if (candidate / "agent_graph.py").exists() and (candidate / "tests").exists():
        PROJECT2_ROOT = candidate
        DAY1_ROOT = candidate.parent
        break
assert DAY1_ROOT is not None and PROJECT2_ROOT is not None, "找不到 day1/project2 项目根目录"
NOTEBOOK_ROOT = PROJECT2_ROOT / "notebooks"
for path in [str(DAY1_ROOT), str(PROJECT2_ROOT), str(NOTEBOOK_ROOT)]:
    if path not in sys.path:
        sys.path.insert(0, path)

from notebook_utils import (
    check,
    check_equal,
    file_inventory,
    load_jsonl,
    masked_environment,
    run_command,
    run_unittest,
    show_markdown,
    show_table,
    source_excerpt,
)

RUN_LIVE_MODEL_TESTS = os.getenv("RUN_LIVE_MODEL_TESTS", "0") == "1"
print(f"Python: {sys.executable}")
print(f"DAY1_ROOT: {DAY1_ROOT}")
print(f"PROJECT2_ROOT: {PROJECT2_ROOT}")
print(f"RUN_LIVE_MODEL_TESTS: {RUN_LIVE_MODEL_TESTS}")

Python: D:\new things\项目1\day1\.venv\Scripts\python.exe
DAY1_ROOT: D:\new things\项目1\day1
PROJECT2_ROOT: D:\new things\项目1\day1\project2
RUN_LIVE_MODEL_TESTS: False


## 为什么需要LangGraph

普通if-else足够做基础回归，所以项目保留`agent_workflow.py`。LangGraph用于显式State、动态工具队列、条件边、checkpoint、人工中断、节点重试和跨进程恢复。

In [2]:
from pathlib import Path
import agent_graph
from handoff_repository import HandoffRepository
from memory_repository import MemoryRepository

temp_dir = tempfile.TemporaryDirectory()
temp_root = Path(temp_dir.name)
saver = agent_graph.create_sqlite_checkpointer(temp_root / "checkpoints.sqlite3")
graph = agent_graph.build_graph(
    saver,
    HandoffRepository(temp_root / "handoff.sqlite3"),
    MemoryRepository(temp_root / "memory.sqlite3"),
)
thread_id = "notebook-langgraph"
first = agent_graph.start_graph_agent(
    "小松PC200原厂液压泵要1件，有没有现货？",
    thread_id=thread_id,
    customer_id="notebook-customer",
    approval_mode="auto",
    graph=graph,
)
snapshot = agent_graph.get_graph_state(thread_id, graph=graph)
show_table([{
    "status": first["status"],
    "turn_count": first["turn_count"],
    "called_tools": first["called_tools"],
    "next": snapshot["next"],
    "messages": len(first["messages"]),
}])
check_equal("首轮完成", first["status"], "completed")
check_equal("库存工具被调用", first["called_tools"], ["inventory_tool"])

D:\new things\项目1\day1\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,status,turn_count,called_tools,next,messages
0,completed,1,[inventory_tool],[],2


[PASS] 首轮完成 | actual='completed', expected='completed'
[PASS] 库存工具被调用 | actual=['inventory_tool'], expected=['inventory_tool']


{'检查项': '库存工具被调用',
 '状态': 'PASS',
 '说明': "actual=['inventory_tool'], expected=['inventory_tool']"}

In [3]:
restarted_saver = agent_graph.create_sqlite_checkpointer(temp_root / "checkpoints.sqlite3")
restarted_graph = agent_graph.build_graph(
    restarted_saver,
    HandoffRepository(temp_root / "handoff.sqlite3"),
    MemoryRepository(temp_root / "memory.sqlite3"),
)
loaded = agent_graph.load_graph_thread(
    thread_id,
    customer_id="notebook-customer",
    graph=restarted_graph,
)
second = agent_graph.start_graph_agent(
    "这个多少钱？",
    thread_id=thread_id,
    customer_id="notebook-customer",
    approval_mode="auto",
    graph=restarted_graph,
)
check_equal("重启后恢复首轮", loaded["turn_count"], 1)
check_equal("同线程继续第二轮", second["turn_count"], 2)
check_equal("第二轮只报价", second["called_tools"], ["quote_tool"])
saver.conn.close()
restarted_saver.conn.close()
temp_dir.cleanup()

[PASS] 重启后恢复首轮 | actual=1, expected=1
[PASS] 同线程继续第二轮 | actual=2, expected=2
[PASS] 第二轮只报价 | actual=['quote_tool'], expected=['quote_tool']


In [4]:
runtime_tests = run_unittest(
    ["tests.test_langgraph_runtime"],
    project2_root=PROJECT2_ROOT,
)
check("LangGraph运行时6条通过", "Ran 6 tests" in runtime_tests.output and "OK" in runtime_tests.output)

$ D:\new things\项目1\day1\.venv\Scripts\python.exe -m unittest tests.test_langgraph_runtime -v
test_checkpoint_persists_and_resumes_in_new_graph (tests.test_langgraph_runtime.LangGraphRuntimeTests.test_checkpoint_persists_and_resumes_in_new_graph) ... ok
test_human_can_edit_arguments_before_approval (tests.test_langgraph_runtime.LangGraphRuntimeTests.test_human_can_edit_arguments_before_approval) ... ok
test_human_can_reject_tool_call (tests.test_langgraph_runtime.LangGraphRuntimeTests.test_human_can_reject_tool_call) ... ok
test_idempotency_record_reuses_existing_result (tests.test_langgraph_runtime.LangGraphRuntimeTests.test_idempotency_record_reuses_existing_result) ... ok
test_non_retryable_error_routes_to_customer_fallback (tests.test_langgraph_runtime.LangGraphRuntimeTests.test_non_retryable_error_routes_to_customer_fallback) ... ok
test_retry_policy_recovers_transient_tool_error (tests.test_langgraph_runtime.LangGraphRuntimeTests.test_retry_policy_recovers_transient_tool_error) .

{'检查项': 'LangGraph运行时6条通过', '状态': 'PASS', '说明': ''}

## State和节点

State保存问题、`thread_id`、`customer_id`、messages、summary、turn_count、槽位、工具队列、结果、错误、审批、图片证据、人工服务单、执行轨迹等。主要节点依次负责加载上下文、图片识别、解析、缺字段判断、选择工具、审批、调用工具、评估接管、写记忆和生成回复。

Conditional edge根据状态决定：继续解析、追问、调用下一个工具、等待审批、等待图片确认、转人工或结束。

### 面试官会问

- 普通if-else为什么不够？什么时候反而更合适？
- State保存什么，哪些数据故意不保存？
- 每个node为什么保持单一职责？
- conditional edge如何避免所有工具节点空转？
- 图失败后如何恢复？checkpoint保存在哪里？
- LangGraph与LangChain Agent有什么区别？

### 参考答案

1. **普通if-else为什么不够，什么时候更合适？** 简单、同步、无暂停恢复的三五步流程用if-else更直接，所以项目保留手写workflow作为回归基线。项目二需要多轮State、动态工具队列、SQLite持久化、审批和人工回复两类interrupt以及跨进程恢复，LangGraph能把这些控制流显式化。
2. **State保存什么，故意不保存什么？** 保存问题、客户/线程标识、messages、摘要、槽位、工具队列、参数、结果、错误、审批、人工服务单和执行轨迹。图片原始二进制、API Key、完整模型Prompt和无关执行日志不放入State，避免checkpoint膨胀、泄密和序列化风险。
3. **为什么node单一职责？** 解析、选工具、审批、执行、接管和回复分开后，每个节点可以独立测试、重试和观察；失败时能知道具体阶段，也不会因为修改回复逻辑而影响工具执行。
4. **conditional edge如何避免空转？** `select_tools`生成`tool_queue`，路由函数只取当前待执行工具；每次完成后推进队列，缺字段、等待审批、等待图片确认和转人工都有独立分支，因此不会固定穿过所有工具节点。
5. **图失败后如何恢复？** 每个步骤按`thread_id`写入SQLite checkpointer。进程重启后用同一`thread_id`读取StateSnapshot，并通过`invoke(None)`或`Command(resume=...)`继续；已成功工具结果由幂等记录复用。单机SQLite已实现，多实例生产环境应换Postgres checkpointer。
6. **LangGraph与LangChain Agent有什么区别？** LangChain Agent偏模型驱动的“思考-选工具-再思考”循环；LangGraph是显式状态图，开发者决定节点、条件边、持久化和中断。本项目把LangChain模型和Tool放进节点，但不让通用Agent绕过业务审批。

**代码落点：** `agent_graph.py::AgentState`、`build_graph`、各route函数、`create_sqlite_checkpointer`和`tests/test_langgraph_runtime.py`。